> **AI-assisted:** this notebook and the code it documents were written with the help of Claude (Anthropic) at the request of a project contributor. Numbers come from running `build_harmonized_dataset.py`; check them before reuse.


# MODIS ↔ VIIRS harmonization — what we did and why
Documentation of how the Terra/Aqua (MODIS) and Suomi-NPP (VIIRS) FIRMS archives were turned into **one consistent hotspot series**
for the burning calendar and the fire-risk model. Region: northern Chihuahua (lat 25.5–31.8, lon −109.5 – −103.3), 2002-07 → 2026-07.

**Output:** `harmonized_hotspots.csv` — 481,029 rows, the 15 original FIRMS columns **plus one flag, `in_calendar`**.
Rows with `in_calendar = 1` (55,902) form the consistent series used for counts, calendars and the risk model; every other row is kept for detail.
The build script is `build_harmonized_dataset.py` (same folder); all numbers below come from running it.

## 1. Inputs (NASA FIRMS archive downloads, CSV)
| File | Sensor | Rows | Dates |
|---|---|---|---|
| `fire_archive_M-C61_848902.csv` | MODIS Collection 6.1, Terra + Aqua (1 km) | 103,050 (Aqua 54,000 · Terra 49,050) | 2002-07-04 → 2026-08-30 |
| `fire_archive_SV-C2_842407.csv` | VIIRS Suomi-NPP (375 m) | 377,979 | 2012-01-20 → 2026-07-31 |

The MODIS archive first arrived as a shapefile; its `.dbf` table was converted to CSV (no values changed).

## 2. Problems that make the two records incomparable
| Problem | MODIS | VIIRS (S-NPP) | Consequence |
|---|---|---|---|
| Pixel size (`scan × track`) | ≈ 1–10 km² (mean ≈ 2) | ≈ 0.14–0.6 km² (mean ≈ 0.24) | VIIRS sees small fires MODIS cannot |
| Detections | 103,050 rows | 377,979 rows | counting rows shows a fake ≈ 10× jump when VIIRS starts (2012) |
| `confidence` | number 0–100 | letters `l`/`n`/`h` | different data type |
| `acq_date` | `20020704` | `2018-04-01` | different format |
| `brightness` (mid-IR) | band 21, saturates ≈ 500 K | band I-4, **saturates at 367 K** | different scale and ceiling |
| Overpass | Terra 10:30, Aqua 13:30 | 13:30 | only Aqua vs VIIRS is like-for-like |
| `instrument` | `MODIS` | `SNPP` | label, not a sensor name |

`bright_t31` (thermal-IR, background temperature) agrees between sensors within 1–3 K, so it is left untouched.

## 3. Method, step by step
**3.1 Same type and format.** `acq_date` → ISO `YYYY-MM-DD`. MODIS `confidence` → `l`/`n`/`h` with the FIRMS thresholds (<30 low, 30–79 nominal, ≥80 high), so both sensors share one scale. Original MODIS scores can be regenerated from the raw file.

**3.2 One afternoon look per day.** Using Terra, Aqua and three VIIRS satellites together makes counts depend on how many satellites flew
(0.60× → 1.22× → 1.83× VIIRS/MODIS across three eras). The series therefore uses **Aqua before 2012-01-20 and Suomi-NPP from then on** (both cross the equator ≈ 13:30).
Terra and Aqua after 2012 stay in the file with `in_calendar = 0`.

**3.3 Make the detection level comparable (the key step).** MODIS only detects a fire above some strength. We measured how often Aqua also detects a fire as a function of the
summed VIIRS FRP inside a MODIS-sized cell (1.4 km, 30 min, matched within 15 min):

| summed VIIRS FRP | 1–2 MW | 8–16 MW | 32–64 MW | 64+ MW |
|---|---|---|---|---|
| P(Aqua also detects it) | 11 % | 43 % | 73 % | 84 % |

The 50 % point is ≈ 15 MW. So S-NPP pixels are merged into 1.4 km cells and a cell counts as a MODIS-equivalent hotspot only if its summed FRP ≥ **15 MW**;
one original row (the highest-FRP pixel) represents the cell. Low-confidence pixels are excluded from the cell sums.

**3.4 Brightness (MODIS rows only).** Brightness temperature cannot be converted pixel by pixel (different bands, pixel dilution, VIIRS saturation), so the correction is a
distribution match fitted on **7,028 Aqua ↔ S-NPP pairs** (same place, ≤ 15 min, day 6,162 / night 866):
`brightness = bright_t31 + f(brightness − bright_t31)`, with `f` a monotonic quantile mapping (separate for day and night), capped at **367 K** (the VIIRS ceiling) and never below `bright_t31`. (Exception: 24 MODIS rows whose own `bright_t31` is 400.1 K keep `brightness = bright_t31`, so they stay above 367 K.)
Mapping the *difference* keeps the physical relation "mid-IR much hotter than thermal-IR ⇒ fire".

**3.5 The flag.** `in_calendar = 1` for (a) non-low Aqua rows before 2012-01-20 and (b) the representative row of each S-NPP cell with summed FRP ≥ 15 MW. Everything else is `0`.
Nothing else is edited: `bright_t31`, `frp`, `scan`, `track`, `version`, `instrument`, `type` are exactly as in the source.

## 4. Validation
**Counts (Aqua vs S-NPP share the 13:30 pass), 175 months 2012-01 → 2026-07**

| S-NPP monthly count | correlation with Aqua | median ratio to Aqua |
|---|---|---|
| raw detections | 0.967 | **11.7×** |
| cells ≥ 15 MW (the series) | **0.984** | **0.91×** |

*Held out:* threshold learned on 2012–2019 only (16.0 MW) and applied to 2020–2026 → correlation 0.986, ratio 0.91. By era the S-NPP/Aqua ratio is 1.12 (2012–17), 1.20 (2018–22), 1.07 (2023–26): **a residual offset of ≈ 10 %**.

**Brightness (fit 2012–2019, test 2020–2026, MODIS vs paired S-NPP)**

| | median gap before → after | distribution mismatch (KS) | mean abs. pixel error |
|---|---|---|---|
| day (n = 2,161) | −17.3 K → **−1.0 K** | 0.58 → 0.13 | 21.5 → 11.1 K |
| night (n = 268) | −18.6 K → +2.5 K | 0.50 → 0.18 | 17.5 → 15.6 K |

No MODIS row ends with brightness < `bright_t31`; 6.2 % of MODIS rows sit at the 367 K cap. Correlation of (brightness − `bright_t31`) with FRP in MODIS: 0.75 → 0.62.

## 5. Check the result on the actual file
Upload `harmonized_hotspots.csv` to the Colab session (or load it from the S3 bucket with `load_s3_csv`) and run the cells below.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt

HARMONIZED_CSV = 'harmonized_hotspots.csv'          # path to the uploaded file
df = pd.read_csv(HARMONIZED_CSV, dtype={'acq_time': str, 'version': str}, parse_dates=['acq_date'])
print(f'{len(df):,} rows, {df.shape[1]} columns  ({df.acq_date.min().date()} -> {df.acq_date.max().date()})')
df['sensor'] = np.where(df['satellite'].isin(['Terra', 'Aqua']), 'MODIS', 'VIIRS')
df.groupby(['satellite', 'in_calendar']).size().unstack(fill_value=0)

In [ ]:
# Same types for both sensors: confidence is l/n/h everywhere, dates are real dates
print(pd.crosstab(df['sensor'], df['confidence']))
print(df.dtypes[['acq_date', 'confidence', 'brightness', 'bright_t31', 'frp', 'in_calendar']])

In [ ]:
# Why the flag exists: naive row counts jump when VIIRS starts (2012); the in_calendar series does not
yearly_all = df[df.confidence != 'l'].groupby(df.acq_date.dt.year).size()
yearly_cal = df[df.in_calendar == 1].groupby(df.acq_date.dt.year).size()
fig, ax = plt.subplots(figsize=(10, 4))
yearly_all.plot(ax=ax, marker='o', label='all non-low rows (naive)'); yearly_cal.plot(ax=ax, marker='o', label='in_calendar = 1 (harmonized)')
ax.axvline(2012, ls='--', c='grey'); ax.set_ylabel('hotspots per year'); ax.set_title('Annual hotspots: naive vs harmonized'); ax.legend(); plt.show()

In [ ]:
# Brightness sanity checks (MODIS rows were mapped, VIIRS rows are original)
m = df[df.sensor == 'MODIS']
print('MODIS rows with brightness < bright_t31:', int((m.brightness < m.bright_t31).sum()))
print('max brightness  MODIS:', m.brightness.max(), ' VIIRS:', df[df.sensor == 'VIIRS'].brightness.max())
print('MODIS rows above 367 K:', int((m.brightness > 367.05).sum()), '(all have bright_t31 = 400.1 K and brightness = bright_t31)')
print(f'MODIS rows at the 367 K cap: {(m.brightness >= 366.95).mean():.1%}')

In [ ]:
# Burning calendar from the harmonized series (month x year), and anomaly vs the same month's median
s = df[df.in_calendar == 1]
cal = s.groupby([s.acq_date.dt.year.rename('year'), s.acq_date.dt.month.rename('month')]).size().unstack(fill_value=0)
anom = cal - cal.median()
fig, ax = plt.subplots(1, 2, figsize=(14, 6))
for a, mat, t, cm in [(ax[0], cal, 'Hotspots per month', 'inferno'), (ax[1], anom, 'Difference from the monthly median', 'RdBu_r')]:
    lim = np.abs(mat.values).max() if cm == 'RdBu_r' else None
    im = a.imshow(mat.values, aspect='auto', cmap=cm, vmin=-lim if lim else None, vmax=lim)
    a.set_yticks(range(len(mat.index))); a.set_yticklabels(mat.index); a.set_xticks(range(12)); a.set_xticklabels(range(1, 13))
    a.set_title(t); a.set_xlabel('month'); plt.colorbar(im, ax=a)
plt.tight_layout(); plt.show()

## 6. Limits (state these in the app)
- Counts are **"MODIS-equivalent" hotspots**, not every fire VIIRS saw; small VIIRS fires remain in the file (`in_calendar = 0`) but are not in the series. Do not use the series for early warning of small fires.
- **≈ 10 % offset** after 2012 (S-NPP/Aqua ratio 1.07–1.20): divide later counts by ≈ 1.1 or warn the user.
- The series ends **2026-07-31** (end of the S-NPP file) — do not compare a partial last month with full months.
- The 15 MW rule and the brightness mapping were fitted in northern Chihuahua only.
- Night brightness rests on 866 pairs: treat as approximate. Values above 367 K are flattened; intensity is better read from `frp`.
- Pre-2012 Aqua brightness uses a mapping fitted on 2012+ (assumes a stable sensor).
- 2011 (7,452 hotspots vs ≈ 1,900/yr before) appears in both Terra and Aqua and peaks in April–June, so it looks like a real fire year, but it is not confirmed externally.

## 7. How this relates to sections 4–6 of this notebook
- **Same goal, different tool.** Section 6 calibrates Aqua against S-NPP with a *monthly count factor* (ΣVIIRS/ΣMODIS per calendar month). The method above instead makes the two
  records comparable at the **detection level** (15 MW cell rule), which also tells which individual VIIRS rows count. The two can be compared: the factor from section 6 on the `in_calendar` series should be close to 1.
- **Same conventions where possible:** single afternoon satellite (S-NPP) for the continuous series, Aqua as the MODIS partner, ISO dates. Satellite names differ: the file uses `SNPP`; map them with
  `df['satellite'] = df['satellite'].replace({'SNPP': 'S-NPP'})` to match `SAT_NAMES`.
- **Not covered here:** the static-source mask for NRT rows (section 3) — the harmonized file uses the archive only, so `type` is always present.